# Exercises week 42

**October 13-17, 2025**

Date: **Deadline is Friday October 17 at midnight**


# Overarching aims of the exercises this week

The aim of the exercises this week is to train the neural network you implemented last week.

To train neural networks, we use gradient descent, since there is no analytical expression for the optimal parameters. This means you will need to compute the gradient of the cost function wrt. the network parameters. And then you will need to implement some gradient method.

You will begin by computing gradients for a network with one layer, then two layers, then any number of layers. Keeping track of the shapes and doing things step by step will be very important this week.

We recommend that you do the exercises this week by editing and running this notebook file, as it includes some checks along the way that you have implemented the neural network correctly, and running small parts of the code at a time will be important for understanding the methods. If you have trouble running a notebook, you can run this notebook in google colab instead(https://colab.research.google.com/drive/1FfvbN0XlhV-lATRPyGRTtTBnJr3zNuHL#offline=true&sandboxMode=true), though we recommend that you set up VSCode and your python environment to run code like this locally.

First, some setup code that you will need.


In [72]:
import autograd.numpy as np  # We need to use this numpy wrapper to make automatic differentiation work later
from autograd import grad, elementwise_grad
from sklearn import datasets
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score


# Defining some activation functions
def ReLU(z):
    return np.where(z > 0, z, 0)


# Derivative of the ReLU function
def ReLU_der(z):
    return np.where(z > 0, 1, 0)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def mse(predict, target):
    return np.mean((predict - target) ** 2)

# Exercise 1 - Understand the feed forward pass

**a)** Complete last weeks' exercises if you haven't already (recommended).


# Exercise 2 - Gradient with one layer using autograd

For the first few exercises, we will not use batched inputs. Only a single input vector is passed through the layer at a time.

In this exercise you will compute the gradient of a single layer. You only need to change the code in the cells right below an exercise, the rest works out of the box. Feel free to make changes and see how stuff works though!


**a)** If the weights and bias of a layer has shapes (10, 4) and (10), what will the shapes of the gradients of the cost function wrt. these weights and this bias be?


The gradients will have the same shape as the matrices themselves. So (10,4) for the weights and 10 for the biases.

**b)** Complete the feed_forward_one_layer function. It should use the sigmoid activation function. Also define the weigth and bias with the correct shapes.


In [73]:
def feed_forward_one_layer(W, b, x):
    z = W @ x + b
    a = sigmoid(z)
    return a


def cost_one_layer(W, b, x, target):
    predict = feed_forward_one_layer(W, b, x)
    return mse(predict, target)


x = np.random.rand(2)
target = np.random.rand(3)

W = np.random.rand(3,2)
b = np.random.rand(3)

**c)** Compute the gradient of the cost function wrt. the weigth and bias by running the cell below. You will not need to change anything, just make sure it runs by defining things correctly in the cell above. This code uses the autograd package which uses backprogagation to compute the gradient!


In [74]:
autograd_one_layer = grad(cost_one_layer, [0, 1])
W_g, b_g = autograd_one_layer(W, b, x, target)
print(W_g, b_g)

[[ 0.00057169  0.01043419]
 [-0.00055742 -0.01017365]
 [ 0.00143358  0.02616475]] [ 0.03546711 -0.0345815   0.08893723]


# Exercise 3 - Gradient with one layer writing backpropagation by hand

Before you use the gradient you found using autograd, you will have to find the gradient "manually", to better understand how the backpropagation computation works. To do backpropagation "manually", you will need to write out expressions for many derivatives along the computation.


We want to find the gradient of the cost function wrt. the weight and bias. This is quite hard to do directly, so we instead use the chain rule to combine multiple derivatives which are easier to compute.

$$
\frac{dC}{dW} = \frac{dC}{da}\frac{da}{dz}\frac{dz}{dW}
$$

$$
\frac{dC}{db} = \frac{dC}{da}\frac{da}{dz}\frac{dz}{db}
$$


**a)** Which intermediary results can be reused between the two expressions?


$\frac{\partial C}{\partial b}$ and $\frac{\partial a}{\partial z}$ can be resued from both parts

**b)** What is the derivative of the cost wrt. the final activation? You can use the autograd calculation to make sure you get the correct result. Remember that we compute the mean in mse.


$\frac{\partial C}{a} = \frac{\partial MSE}{\partial a}= \partial \frac{\partial \frac{1}{n} (target-a)^T(target-a)}{\partial a}$

We use our previous result $\frac{\partial (x-As)^T(x-As)}{\partial a}=-2(x-As)^TA$ and we get $\frac{\partial C}{\partial a}=-\frac{2}{n}(target-a)^T$

In [75]:
z = W @ x + b
a = sigmoid(z)

predict = a


def mse_der(predict, target):
    return -2/len(predict)*np.transpose(target-predict)


print(mse_der(predict, target))

cost_autograd = grad(mse, 0)
print(cost_autograd(predict, target))

[ 0.20629544 -0.1747271   0.43649812]
[ 0.20629544 -0.1747271   0.43649812]


**c)** What is the expression for the derivative of the sigmoid activation function? You can use the autograd calculation to make sure you get the correct result.


In [76]:
def sigmoid_der(z):
    return np.exp(-z)/((1+np.exp(-z))**2)


print(sigmoid_der(z))

sigmoid_autograd = elementwise_grad(sigmoid, 0)
print(sigmoid_autograd(z))

[0.17192385 0.19791723 0.20375169]
[0.17192385 0.19791723 0.20375169]


**d)** Using the two derivatives you just computed, compute this intermetidary gradient you will use later:

$$
\frac{dC}{dz} = \frac{dC}{da}\frac{da}{dz}
$$


In [77]:
dC_da = mse_der(predict,target)
dC_dz = dC_da*sigmoid_der(z)
print(dC_dz)

[ 0.03546711 -0.0345815   0.08893723]


**e)** What is the derivative of the intermediary z wrt. the weight and bias? What should the shapes be? The one for the weights is a little tricky, it can be easier to play around in the next exercise first. You can also try computing it with autograd to get a hint.


Notation gotten from my co student Ådne Rostad:


$z_k = \left ( \sum_l W_{kl}x_l \right ) + b_k$

We take the derivative in respect to the weights
$\frac{\partial{\sum_{l}(W_{kl}*x_{l}})+b_k}{\partial W_{ij}}=\sum_l \delta_{ki}\delta_{lj}x_l=\delta{ki}{x_j}$

We take the derivatives in respect to the bias
$\frac{\partial (\sum_{l}W_{kl}x_l)+b_k}{\partial b_k}=\frac{\partial b_k}{\partial b_k}=1$

**f)** Now combine the expressions you have worked with so far to compute the gradients! Note that you always need to do a feed forward pass while saving the zs and as before you do backpropagation, as they are used in the derivative expressions


In [78]:
dC_da = mse_der(predict,target)
dC_dz = dC_da * sigmoid_der(z)
dC_dW = np.outer(dC_dz,x)
dC_db = dC_dz

print(dC_dW, dC_db)

[[ 0.00057169  0.01043419]
 [-0.00055742 -0.01017365]
 [ 0.00143358  0.02616475]] [ 0.03546711 -0.0345815   0.08893723]


You should get the same results as with autograd.


In [79]:
W_g, b_g = autograd_one_layer(W, b, x, target)
print(W_g, b_g)

[[ 0.00057169  0.01043419]
 [-0.00055742 -0.01017365]
 [ 0.00143358  0.02616475]] [ 0.03546711 -0.0345815   0.08893723]


# Exercise 4 - Gradient with two layers writing backpropagation by hand


Now that you have implemented backpropagation for one layer, you have found most of the expressions you will need for more layers. Let's move up to two layers.


In [80]:
x = np.random.rand(2)
target = np.random.rand(4)

W1 = np.random.rand(3, 2)
b1 = np.random.rand(3)

W2 = np.random.rand(4, 3)
b2 = np.random.rand(4)

layers = [(W1, b1), (W2, b2)]

In [81]:
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

We begin by computing the gradients of the last layer, as the gradients must be propagated backwards from the end.

**a)** Compute the gradients of the last layer, just like you did the single layer in the previous exercise.


In [82]:
dC_da2 = mse_der(a2,target)
dC_dz2 = dC_da2*sigmoid_der(z2)
dC_dW2 = np.outer(dC_dz2,x)
dC_db2 = dC_dz2

To find the derivative of the cost wrt. the activation of the first layer, we need a new expression, the one furthest to the right in the following.

$$
\frac{dC}{da_1} = \frac{dC}{dz_2}\frac{dz_2}{da_1}
$$

**b)** What is the derivative of the second layer intermetiate wrt. the first layer activation? (First recall how you compute $z_2$)

$$
\frac{dz_2}{da_1}
$$


We have that $z_2=W_2a_1+b_2$
$$
\frac{\partial z_2}{\partial a_1}=W_2
$$


**c)** Use this expression, together with expressions which are equivelent to ones for the last layer to compute all the derivatives of the first layer.

$$
\frac{dC}{dW_1} = \frac{dC}{da_1}\frac{da_1}{dz_1}\frac{dz_1}{dW_1}
$$

$$
\frac{dC}{db_1} = \frac{dC}{da_1}\frac{da_1}{dz_1}\frac{dz_1}{db_1}
$$


In [83]:
dC_da1 = dC_dz2 @ W2
dC_dz1 = dC_da1 * sigmoid_der(z1)
dC_dW1 = np.outer(dC_dz1, x)
dC_db1 = dC_dz1

In [84]:
print(dC_dW1, dC_db1)
print(dC_dW2, dC_db2)

[[0.0004105  0.00193953]
 [0.00130598 0.00617048]
 [0.00083317 0.00393656]] [0.00418815 0.01332431 0.00850045]
[[0.00119348 0.00563896]
 [0.0015806  0.00746802]
 [0.00092082 0.00435069]
 [0.00387431 0.01830533]] [0.01217657 0.01612617 0.00939473 0.03952786]


**d)** Make sure you got the same gradient as the following code which uses autograd to do backpropagation.


In [85]:
def feed_forward_two_layers(layers, x):
    W1, b1 = layers[0]
    z1 = W1 @ x + b1
    a1 = sigmoid(z1)

    W2, b2 = layers[1]
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)

    return a2

In [86]:
def cost_two_layers(layers, x, target):
    predict = feed_forward_two_layers(layers, x)
    return mse(predict, target)


grad_two_layers = grad(cost_two_layers, 0)
grad_two_layers(layers, x, target)

[(array([[0.0004105 , 0.00193953],
         [0.00130598, 0.00617048],
         [0.00083317, 0.00393656]]),
  array([0.00418815, 0.01332431, 0.00850045])),
 (array([[0.00918055, 0.00758727, 0.00971728],
         [0.01215837, 0.01004828, 0.01286918],
         [0.00708318, 0.00585389, 0.00749728],
         [0.02980212, 0.02462997, 0.03154446]]),
  array([0.01217657, 0.01612617, 0.00939473, 0.03952786]))]

**e)** How would you use the gradient from this layer to compute the gradient of an even earlier layer? Would the expressions be any different?


We could write 

$$
\frac{dC}{dW_0} = \frac{dC}{da_0}\frac{da_0}{dz_0}\frac{dz_0}{dW_0}
$$

$$
\frac{dC}{db_0} = \frac{dC}{da_0}\frac{da_0}{dz_0}\frac{dz_0}{db_0}
$$

$$
\frac{dC}{da_0} = \frac{dC}{dz_1}\frac{dz_1}{da_0}
$$
 
Where we can use dC/dz_1 and d_z_1 / da_0 # im sick of writing latex




# Exercise 5 - Gradient with any number of layers writing backpropagation by hand


Well done on getting this far! Now it's time to compute the gradient with any number of layers.

First, some code from the general neural network code from last week. Note that we are still sending in one input vector at a time. We will change it to use batched inputs later.


In [87]:
def create_layers(network_input_size, layer_output_sizes):
    layers = []

    i_size = network_input_size
    for layer_output_size in layer_output_sizes:
        W = np.random.randn(layer_output_size, i_size)
        b = np.random.randn(layer_output_size)
        layers.append((W, b))

        i_size = layer_output_size
    return layers


def feed_forward(input, layers, activation_funcs):
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        z = W @ a + b
        a = activation_func(z)
    return a


def cost(layers, input, activation_funcs, target):
    predict = feed_forward(input, layers, activation_funcs)
    return mse(predict, target)

You might have already have noticed a very important detail in backpropagation: You need the values from the forward pass to compute all the gradients! The feed forward method above is great for efficiency and for using autograd, as it only cares about computing the final output, but now we need to also save the results along the way.

Here is a function which does that for you.


In [88]:
def feed_forward_saver(input, layers, activation_funcs):
    layer_inputs = []
    zs = []
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        layer_inputs.append(a)
        z = W @ a + b
        a = activation_func(z)

        zs.append(z)

    return layer_inputs, zs, a

**a)** Now, complete the backpropagation function so that it returns the gradient of the cost function wrt. all the weigths and biases. Use the autograd calculation below to make sure you get the correct answer.


In [89]:
def backpropagation(
    input, layers, activation_funcs, target, activation_ders, cost_der=mse_der
):
    layer_inputs, zs, predict = feed_forward_saver(input, layers, activation_funcs)

    layer_grads = [() for layer in layers]

    dC_dz_iter = None

    # We loop over the layers, from the last to the first
    for i in reversed(range(len(layers))):
        layer_input, z, activation_der = layer_inputs[i], zs[i], activation_ders[i]

        if i == len(layers) - 1:
            # For last layer we use cost derivative as dC_da(L) can be computed directly
            dC_da = cost_der(predict, target)
        else:
            # For other layers we build on previous z derivative, as dC_da(i) = dC_dz(i+1) * dz(i+1)_da(i)
            (W, b) = layers[i + 1]
            dC_da = np.transpose(W) @ dC_dz_iter

        dC_dz = dC_da*activation_der(z)
        dC_dW = np.outer(dC_dz,layer_input)
        dC_db = dC_dz

        layer_grads[i] = (dC_dW, dC_db)

        dC_dz_iter = dC_dz

    return layer_grads

In [90]:
network_input_size = 2
layer_output_sizes = [3, 4]
activation_funcs = [sigmoid, ReLU]
activation_ders = [sigmoid_der, ReLU_der]

layers = create_layers(network_input_size, layer_output_sizes)

x = np.random.rand(network_input_size)
target = np.random.rand(4)

In [91]:
layer_grads = backpropagation(x, layers, activation_funcs, target, activation_ders)
print(layer_grads)

[(array([[ 0.00520773,  0.00011043],
       [ 0.0292254 ,  0.0006197 ],
       [-0.01275656, -0.00027049]]), array([ 0.00738723,  0.04145656, -0.01809533])), (array([[-0.36875691, -0.08635674, -0.27581744],
       [-0.        , -0.        , -0.        ],
       [-0.        , -0.        , -0.        ],
       [-0.        , -0.        , -0.        ]]), array([-0.40370985, -0.        , -0.        , -0.        ]))]


In [92]:
cost_grad = grad(cost, 0)
cost_grad(layers, x, [sigmoid, ReLU], target)

[(array([[ 0.00520773,  0.00011043],
         [ 0.0292254 ,  0.0006197 ],
         [-0.01275656, -0.00027049]]),
  array([ 0.00738723,  0.04145656, -0.01809533])),
 (array([[-0.36875691, -0.08635674, -0.27581744],
         [ 0.        ,  0.        ,  0.        ],
         [ 0.        ,  0.        ,  0.        ],
         [ 0.        ,  0.        ,  0.        ]]),
  array([-0.40370985,  0.        ,  0.        ,  0.        ]))]

# Exercise 6 - Batched inputs

Make new versions of all the functions in exercise 5 which now take batched inputs instead. See last weeks exercise 5 for details on how to batch inputs to neural networks. You will also need to update the backpropogation function.


In [96]:
def create_layers_batch(network_input_size, layer_output_sizes):
    layers = []

    i_size = network_input_size
    for layer_output_size in layer_output_sizes:
        W = np.random.randn(i_size,layer_output_size)
        b = np.random.randn(1,layer_output_size)
        layers.append((W, b))

        i_size = layer_output_size
    return layers


def feed_forward_batch(input, layers, activation_funcs):
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        z = a @ W + b
        a = activation_func(z)
    return a


def cost_batch(layers, input, activation_funcs, target):
    predict = feed_forward(input, layers, activation_funcs)
    return mse(predict, target)


def feed_forward_saver_batch(input, layers, activation_funcs):
    layer_inputs = []
    zs = []
    a = input
    for (W, b), activation_func in zip(layers, activation_funcs):
        layer_inputs.append(a)
        z = a @ W + b
        a = activation_func(z)

        zs.append(z)

    return layer_inputs, zs, a


def backpropagation_batch(
    input, layers, activation_funcs, target, activation_ders, cost_der=mse_der
):
    layer_inputs, zs, predict = feed_forward_saver_batch(input, layers, activation_funcs)

    layer_grads = [() for layer in layers]

    dC_dz_iter = None

    # We loop over the layers, from the last to the first
    for i in reversed(range(len(layers))):
        layer_input, z, activation_der = layer_inputs[i], zs[i], activation_ders[i]

        if i == len(layers) - 1:
            # For last layer we use cost derivative as dC_da(L) can be computed directly
            dC_da = cost_der(predict, target)
        else:
            # For other layers we build on previous z derivative, as dC_da(i) = dC_dz(i+1) * dz(i+1)_da(i)
            (W, b) = layers[i + 1]
            dC_da = dC_dz_iter @ np.transpose(W) 

        dC_dz = dC_da*activation_der(z)
        dC_dW = np.transpose(layer_input)@dC_dz
        dC_db = np.sum(dC_dz,axis=0)

        layer_grads[i] = (dC_dW, dC_db)

        dC_dz_iter = dC_dz

    return layer_grads

# Exercise 7 - Training


**a)** Complete exercise 6 and 7 from last week, but use your own backpropogation implementation to compute the gradient.
- IMPORTANT: Do not implement the derivative terms for softmax and cross-entropy separately, it will be very hard!
- Instead, use the fact that the derivatives multiplied together simplify to **prediction - target** (see [source1](https://medium.com/data-science/derivative-of-the-softmax-function-and-the-categorical-cross-entropy-loss-ffceefc081d1), [source2](https://shivammehta25.github.io/posts/deriving-categorical-cross-entropy-and-softmax/))



In [97]:
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)   
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)


def softmax_cross_entropy_der(pred, target):
    B = pred.shape[0]
    return (pred - target) / B


def cross_entropy(predict, target):
    return np.sum(-target * np.log(predict))


def cost(input, layers, activation_funcs, target):
    predict = feed_forward_batch(input, layers, activation_funcs)
    return cross_entropy(predict, target)

def accuracy(predictions, targets):
    one_hot_predictions = np.zeros(predictions.shape)

    for i, prediction in enumerate(predictions):
        one_hot_predictions[i, np.argmax(prediction)] = 1
    return accuracy_score(one_hot_predictions, targets)

def predict(X_):
    a = X_
    for (W,b), act in zip(layers, activation_funcs):
        a = act(a @ W + b)
    return a 



In [101]:
iris = datasets.load_iris()

inputs = iris.data

# Since each prediction is a vector with a score for each of the three types of flowers,
# we need to make each target a vector with a 1 for the correct flower and a 0 for the others.
targets = np.zeros((len(iris.data), 3))
X = iris.data.astype(float)
y = iris.target

#one hot encoding 
Y = np.zeros((len(y),3))
Y[np.arange(len(y)),y] = 1

X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-14)

network_input_size = 4
layer_output_sizes = [8,4,3]
activation_funcs = [sigmoid,ReLU,softmax]
layers = create_layers_batch(network_input_size, layer_output_sizes)

activation_ders = [sigmoid_der,ReLU_der, lambda z: np.ones_like(z)]

lr = 0.05
epochs = 300
batch_size = 32
n = X.shape[0]

for epoch in range(1, epochs + 1):
    idx = np.random.permutation(n)
    Xs, Ys = X[idx], Y[idx]

    for start in range(0, n, batch_size):
        Xb = Xs[start:start+batch_size]
        Yb = Ys[start:start+batch_size]

        grads = backpropagation_batch(
            Xb,
            layers,
            activation_funcs, 
            Yb,
            activation_ders,
            cost_der=softmax_cross_entropy_der
        )
        
        new_layers = []
        for (W,b), (dW,db) in zip(layers, grads):
            W_new = W - lr * dW
            b_new = b - lr * db
            new_layers.append((W_new, b_new))
        layers = new_layers

    #prinouts 
    if epoch % 50 == 0 or epoch == 1:
        preds = predict(X)
        loss = cross_entropy(preds, Y)
        acc = accuracy(preds, Y)
        print(f"On epoch {epoch} the loss was: {loss} and accuracy was: {acc}")



On epoch 1 the loss was: 369.303821519669 and accuracy was: 0.3333333333333333
On epoch 50 the loss was: 64.06492905093593 and accuracy was: 0.84
On epoch 100 the loss was: 48.89856640585257 and accuracy was: 0.8733333333333333
On epoch 150 the loss was: 36.541188255259264 and accuracy was: 0.9133333333333333
On epoch 200 the loss was: 22.647245035787236 and accuracy was: 0.96
On epoch 250 the loss was: 14.34411896617547 and accuracy was: 0.9666666666666667
On epoch 300 the loss was: 10.978110730336464 and accuracy was: 0.9733333333333334


Yippie we learn. No proper splitting of train test though, sorry.

**b)** Use stochastic gradient descent with momentum when you train your network.


In [103]:
iris = datasets.load_iris()

inputs = iris.data

# Since each prediction is a vector with a score for each of the three types of flowers,
# we need to make each target a vector with a 1 for the correct flower and a 0 for the others.
targets = np.zeros((len(iris.data), 3))
X = iris.data.astype(float)
y = iris.target

#one hot encoding 
Y = np.zeros((len(y),3))
Y[np.arange(len(y)),y] = 1

X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-14)

network_input_size = 4
layer_output_sizes = [8,4,3]
activation_funcs = [sigmoid,ReLU,softmax]
layers = create_layers_batch(network_input_size, layer_output_sizes)

activation_ders = [sigmoid_der,ReLU_der, lambda z: np.ones_like(z)]

lr = 0.05
epochs = 300
batch_size = 32
n = X.shape[0]
beta = 0.8
velocities = [(np.zeros_like(W), np.zeros_like(b)) for (W, b) in layers]


for epoch in range(1, epochs + 1):
    idx = np.random.permutation(n)
    Xs, Ys = X[idx], Y[idx]

    for start in range(0, n, batch_size):
        Xb = Xs[start:start+batch_size]
        Yb = Ys[start:start+batch_size]

        grads = backpropagation_batch(
            Xb,
            layers,
            activation_funcs, 
            Yb,
            activation_ders,
            cost_der=softmax_cross_entropy_der
        )
        
        new_layers = []
        new_vels = []
        for (W, b), (dW, db), (vW, vB) in zip(layers, grads, velocities):
            vW_new = beta * vW - lr * dW
            vB_new = beta * vB - lr * db
            W_new = W + vW_new
            b_new = b + vB_new
            new_layers.append((W_new, b_new))
            new_vels.append((vW_new, vB_new))
        layers = new_layers
        velocities = new_vels

    #prinouts 
    if epoch % 50 == 0 or epoch == 1:
        preds = predict(X)
        loss = cross_entropy(preds, Y)
        acc = accuracy(preds, Y)
        print(f"On epoch {epoch} the loss was: {loss} and accuracy was: {acc}")



On epoch 1 the loss was: 144.15406709120793 and accuracy was: 0.6933333333333334
On epoch 50 the loss was: 11.628836482312632 and accuracy was: 0.96
On epoch 100 the loss was: 7.924924284068939 and accuracy was: 0.98
On epoch 150 the loss was: 6.904277140246527 and accuracy was: 0.9866666666666667
On epoch 200 the loss was: 6.477330627450321 and accuracy was: 0.9866666666666667
On epoch 250 the loss was: 6.31935795591804 and accuracy was: 0.9866666666666667
On epoch 300 the loss was: 5.994662915399417 and accuracy was: 0.9866666666666667


Training is much quicker

# Exercise 8 (Optional) - Object orientation

Passing in the layers, activations functions, activation derivatives and cost derivatives into the functions each time leads to code which is easy to understand in isoloation, but messier when used in a larger context with data splitting, data scaling, gradient methods and so forth. Creating an object which stores these values can lead to code which is much easier to use.

**a)** Write a neural network class. You are free to implement it how you see fit, though we strongly recommend to not save any input or output values as class attributes, nor let the neural network class handle gradient methods internally. Gradient methods should be handled outside, by performing general operations on the layer_grads list using functions or classes separate to the neural network.

We provide here a skeleton structure which should get you started.


In [ ]:
class NeuralNetwork:
    def __init__(
        self,
        network_input_size,
        layer_output_sizes,
        activation_funcs,
        activation_ders,
        cost_fun,
        cost_der,
    ):
        pass

    def predict(self, inputs):
        # Simple feed forward pass
        pass

    def cost(self, inputs, targets):
        pass

    def _feed_forward_saver(self, inputs):
        pass

    def compute_gradient(self, inputs, targets):
        pass

    def update_weights(self, layer_grads):
        pass

    # These last two methods are not needed in the project, but they can be nice to have! The first one has a layers parameter so that you can use autograd on it
    def autograd_compliant_predict(self, layers, inputs):
        pass

    def autograd_gradient(self, inputs, targets):
        pass